# Scalable ML Lab 1 — 本地 GPU / Colab 双环境启动与 Smoke Test

本 Notebook 自动适配 **本地 GPU（推荐 `.venv`）** 与 **Google Colab（T4 GPU）**，一键完成：
1. 自动识别本地仓库或拉取公开 GitHub 仓库 `https://github.com/Mrpooool/ScalableMlLab.git` 并挂载 `lab1/src/`
2. 下载 CIFAR-10（高速镜像）并生成/核对固定的 `45,000 / 5,000` 划分 (`SPLIT_SEED=2223`)
3. 检查手写 Part A9 **AlexNet** 与 **ResNet-18** 的各阶段特征图尺寸与参数量
4. 运行 **1-epoch Smoke Test**（验证训练、确定性 `train_eval`、峰值显存、推理测速与标准 JSON 落盘）

> **环境提示**：
> - **本地运行（VS Code / Cursor）**：直接选择仓库根目录 `.venv` 作为 Python 内核，全部运行即可；数据与权重保存在已 `.gitignore` 忽略的 `lab1/data/` 与 `lab1/checkpoints/`。
> - **网页版 Colab 运行**：切换运行时为 **T4 GPU**，保持 `USE_DRIVE_IN_COLAB = True` 挂载共享云盘（[共享云盘入口](https://drive.google.com/drive/folders/1eRebAfEqnmBqUf3MgwX3NlcAFC54Izzp?usp=sharing)）。
> - **VS Code Colab 插件运行**：将 `USE_DRIVE_IN_COLAB = False`（避免 `drive.mount` 弹窗卡死），使用云端本地 `/content/work_lab1`。

In [ ]:
# 1. 自动识别本地环境 vs Google Colab，并初始化路径与设备
from pathlib import Path
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
USE_DRIVE_IN_COLAB = True  # VS Code Colab 插件中请设为 False
DRIVE_FOLDER_NAME = "ScalableMlLab"
SHARED_DRIVE_URL = "https://drive.google.com/drive/folders/1eRebAfEqnmBqUf3MgwX3NlcAFC54Izzp?usp=sharing"
REPO_URL = "https://github.com/Mrpooool/ScalableMlLab.git"
BRANCH = "main"

if IN_COLAB:
    REPO_DIR = Path("/content/ScalableMlLab")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

    if USE_DRIVE_IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
        if not DRIVE_ROOT.exists():
            raise FileNotFoundError(
                f"未在 MyDrive 下找到 '{DRIVE_FOLDER_NAME}'，请先打开 {SHARED_DRIVE_URL} 添加快捷方式到 My Drive。"
            )
        WORK_LAB1 = DRIVE_ROOT / "lab1"
    else:
        WORK_LAB1 = Path("/content/work_lab1")
else:
    # 本地运行：自动向上查找仓库根目录（包含 lab1/src）
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    REPO_DIR = next((p for p in candidates if (p / "lab1" / "src").exists()), cwd)
    WORK_LAB1 = REPO_DIR / "lab1"

for sub in ("data", "checkpoints", "results", "configs"):
    (WORK_LAB1 / sub).mkdir(parents=True, exist_ok=True)

SRC_DIR = REPO_DIR / "lab1" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(REPO_DIR)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if platform.system() == "Windows" else 2

print("Mode:", "Colab" if IN_COLAB else "Local")
print("Repo dir:", REPO_DIR)
print("Work dir:", WORK_LAB1)
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "None")
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| num_workers:", NUM_WORKERS)

In [ ]:
# 2. 加载 CIFAR-10 并生成/核对固定 train/val 划分 (45,000 / 5,000, SPLIT_SEED=2223)
import shutil
import pandas as pd

from data import CIFAR10_CLASSES, get_or_create_split, load_cifar10_datasets, make_loaders
from io_utils import load_json

train_ds, test_ds = load_cifar10_datasets(WORK_LAB1 / "data")

SPLIT_PATH = REPO_DIR / "lab1" / "configs" / "split_indices.json"
split_info = get_or_create_split(train_ds, SPLIT_PATH)

# 如果 WORK_LAB1 与仓库目录不同（例如 Colab + Drive），同步备份一份
work_split = WORK_LAB1 / "configs" / "split_indices.json"
if work_split.resolve() != SPLIT_PATH.resolve():
    shutil.copy2(SPLIT_PATH, work_split)

stats = split_info["stats"]
train_ids = split_info["train_ids"]
val_ids = split_info["val_ids"]

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ds)}")
display(pd.DataFrame(stats, index=["red", "green", "blue"]))

In [ ]:
# 3. 验证手写 Part A9 AlexNet 与 ResNet-18 的特征图尺寸与参数量
from models import build_model, count_parameters, trace_shapes

for model_name in ("alexnet_cifar", "resnet18_cifar"):
    model, input_size = build_model(model_name, device=device)
    params = count_parameters(model)
    shapes = trace_shapes(model, input_size=input_size, device=device)
    print(f"\n=== {model_name} (input={input_size}x{input_size}) ===")
    print("Parameters:", params)
    display(pd.DataFrame(shapes))
    del model

In [ ]:
# 4. 运行 1-epoch Smoke Test（仅验证全链路，不评测 test 集，不污染正式结果）
from train import ExperimentConfig, run_experiment

SMOKE_RESULTS_DIR = WORK_LAB1 / "results" / "smoke_test"
SMOKE_CKPT_DIR = WORK_LAB1 / "checkpoints" / "smoke_test"
SMOKE_MANIFEST = SMOKE_RESULTS_DIR / "checkpoints_manifest.json"

baseline_cfg = load_json(REPO_DIR / "lab1" / "configs" / "baseline.json")
loaders = make_loaders(
    train_dataset=train_ds,
    test_dataset=test_ds,
    train_ids=train_ids,
    val_ids=val_ids,
    batch_size=baseline_cfg["batch_size"],
    seed=baseline_cfg["default_seed"],
    num_workers=NUM_WORKERS,
)

smoke_summaries = []
for model_name in ("alexnet_cifar", "resnet18_cifar"):
    mcfg = baseline_cfg["baselines"][model_name]
    cfg = ExperimentConfig(
        exp_id=f"smoke_{model_name}_e1_s42",
        task="phase1_smoke",
        owner="role_a",
        model=model_name,
        model_options=mcfg["model_options"],
        seed=baseline_cfg["default_seed"],
        epochs=1,
        batch_size=baseline_cfg["batch_size"],
        optimizer=mcfg["optimizer"],
        lr=mcfg["lr"],
        momentum=mcfg["momentum"],
        weight_decay=mcfg["weight_decay"],
        scheduler=mcfg["scheduler"],
        notes="Phase 1 1-epoch smoke test",
    )
    _, record, _ = run_experiment(
        config=cfg,
        loaders=loaders,
        stats=stats,
        device=device,
        results_dir=SMOKE_RESULTS_DIR,
        checkpoint_dir=SMOKE_CKPT_DIR,
        manifest_path=SMOKE_MANIFEST,
        drive_rel_prefix="checkpoints/smoke_test",
        evaluate_test=False,
        run_benchmark=True,
        repo_root=REPO_DIR,
    )
    smoke_summaries.append(
        {
            "exp_id": record["exp_id"],
            "device": record["environment"]["device_name"],
            "params": record["model_stats"]["total_params"],
            "epoch_sec": record["training"]["avg_epoch_time_sec"],
            "peak_mem_mb": record["training"]["peak_gpu_mem_mb"],
            "val_acc_e1": round(record["training"]["best_val_acc"], 4),
            "bs1_ms": round(record["inference_benchmark"]["bs1_latency_ms"], 3),
            "bs128_imgs_s": round(record["inference_benchmark"]["bs128_throughput_imgs_per_sec"], 1),
        }
    )

display(pd.DataFrame(smoke_summaries))
print("Smoke test JSONs saved under:", SMOKE_RESULTS_DIR)